# Task 6 · SFT only
Clone the shared semeval repository and install the package in `task 6/`.
No code ZIP or local dataset upload. Edit REPO_URL / GIT_REF in the cell below.
Use Runtime → Change runtime type → GPU for training. The transfer notebook can run on CPU.


In [ ]:
# Repository settings: editable in the notebook.
REPO_URL = 'https://github.com/iamtro-2006/semeval.git'
GIT_REF = 'main'  # branch, tag, or commit
CHECKOUT = '/content/semeval'
PROJECT_SUBDIR = 'task 6'

from pathlib import Path
import os, subprocess, sys
checkout = Path(CHECKOUT)
if not checkout.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(checkout)], check=True)
else:
    actual = subprocess.check_output(['git', '-C', str(checkout), 'remote', 'get-url', 'origin'], text=True).strip()
    if actual != REPO_URL:
        raise ValueError('CHECKOUT belongs to a different repository; choose another CHECKOUT')
    dirty = subprocess.check_output(['git', '-C', str(checkout), 'status', '--porcelain'], text=True)
    if dirty.strip():
        raise RuntimeError('Checkout has edits; preserve your changes or choose a fresh CHECKOUT before switching revision')
subprocess.run(['git', '-C', str(checkout), 'fetch', 'origin', GIT_REF], check=True)
subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
os.chdir(checkout / PROJECT_SUBDIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation', '-e', '.'], check=True)
print('Project:', Path.cwd())
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## Install GPU environment
Restart the runtime if requested, then rerun clone/setup. GPU training has not been run on the CPU-only development machine.


In [ ]:
%pip -q install -r requirements.txt
!python -m pip check
!nvidia-smi


## MMCQA folder and authentication
Only ROOT_FOLDER_ID selects storage. Paste the ID or URL of your MMCQA folder.
RCLONE_CONFIG_PATH points to your authenticated rclone.conf; this is not a storage root.
All data, adapters, checkpoints and results are addressed relative to the selected ID.
If your config is on mounted Drive, run the optional mount cell first.


In [ ]:
# Optional, only to access a config file stored on Drive:
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
ROOT_FOLDER_ID = ''  # REQUIRED: ID or URL of the MMCQA folder
RCLONE_CONFIG_PATH = '/content/rclone.conf'
REMOTE = ''  # auto-detect when the config contains one Google Drive remote


In [ ]:
!sudo apt-get -qq update && sudo apt-get -qq install -y rclone
import configparser, re, tempfile
from urllib.parse import urlparse
config_source = Path(RCLONE_CONFIG_PATH).expanduser()
if not config_source.is_file():
    raise FileNotFoundError(f'Upload your authenticated rclone.conf or mount Drive: {config_source}')
sections = configparser.RawConfigParser()
sections.read(config_source, encoding='utf-8-sig')
remotes = [name for name in sections.sections() if sections[name].get('type') == 'drive']
if not REMOTE:
    if len(remotes) != 1:
        raise ValueError(f'Select REMOTE from Google Drive sections: {remotes}')
    REMOTE = remotes[0]
if REMOTE not in remotes:
    raise ValueError('REMOTE must be a Google Drive section')
root_id = ROOT_FOLDER_ID.strip()
if root_id.startswith(('https://', 'http://')):
    match = re.search(r'/folders/([^/?]+)', urlparse(root_id).path)
    root_id = match.group(1) if match else ''
if not re.fullmatch(r'[A-Za-z0-9_-]+', root_id):
    raise ValueError('Enter the ID or folder URL of MMCQA in ROOT_FOLDER_ID')
temporary_config = tempfile.TemporaryDirectory(prefix='mmcqa-rclone-')
config_path = Path(temporary_config.name) / 'rclone.conf'
config_path.write_bytes(config_source.read_bytes())
config_path.chmod(0o600)
rclone_base = ['rclone', '--config', str(config_path), '--drive-root-folder-id', root_id]
def cloud(relative):
    if relative.startswith('/') or '..' in Path(relative).parts:
        raise ValueError('Storage paths must be relative to MMCQA')
    return f'{REMOTE}:{relative}'
def copy(source, destination, *extra):
    subprocess.run(rclone_base + ['copy', str(source), str(destination),
                   '--transfers', '4', '--checkers', '8', '--progress', *extra], check=True)
subprocess.run(rclone_base + ['lsd', cloud(''), '--max-depth', '1'], check=True)
print('All storage is relative to MMCQA folder ID:', root_id)


## Prepare data on Colab SSD
The download location is automatically derived from ROOT_FOLDER_ID.


In [ ]:
USE_DRIVE_DATA = True
if USE_DRIVE_DATA:
    copy(cloud('task6-data/raw'), 'data/raw')
else:
    subprocess.run([sys.executable, '-m', 'mmcultureqa.data.prepare', 'download'], check=True)
subprocess.run([sys.executable, '-m', 'mmcultureqa.data.prepare', 'prepare'], check=True)


## Experiment settings
Only ROOT_FOLDER_ID determines cloud storage. RUN_NAME groups your experiments under it.
Intermediate checkpoints stay on local SSD; only the selected final adapter uploads after successful training.
Local SSD is used for training and inference. If the runtime is lost before the final upload, local checkpoints cannot be recovered from Drive.
Default: native BF16 LoRA, CUDA FP16 mmBERT; GRPO uses vLLM colocate with sleep enabled.
SFT's optional in-training generation monitor uses the native ms-swift Transformers path.
Final validation/test inference uses vLLM. This is an unbenchmarked 24 GB starting configuration.


In [ ]:
import json
USE_4BIT = False
PREDICT_WITH_GENERATE = False  # default SFT: val loss + token accuracy; True: generation + mmBERT
RUN_FULL = True  # full training from YAML; False explicitly selects a one-step smoke run
RUN_NAME = 'experiment01'
if not re.fullmatch(r'[A-Za-z0-9_-]+', RUN_NAME):
    raise ValueError('RUN_NAME must contain only letters, numbers, underscores or hyphens')
RUNS = Path('outputs/runs') / RUN_NAME
RUNS.mkdir(parents=True, exist_ok=True)
SFT_STAGE = 'sft' if RUN_FULL else 'sft-smoke'
GRPO_STAGE = 'grpo' if RUN_FULL else 'grpo-smoke'
SFT = str(RUNS / SFT_STAGE)
GRPO = str(RUNS / GRPO_STAGE)
common = {'model.dtype': 'bfloat16', 'model.load_in_4bit': USE_4BIT,
          'training.predict_with_generate': PREDICT_WITH_GENERATE}

def upload_run(local_output, cloud_subdir):
    # Never upload intermediate checkpoints, including during final validation sync.
    copy(local_output, cloud(cloud_subdir),
         '--exclude', '/checkpoint-*', '--exclude', '/checkpoint-*/**',
         '--exclude', '/swift_data/**')

def run(module, config, overrides, local_output, cloud_subdir):
    cmd = [sys.executable, '-m', module, '--config', config]
    for key, value in overrides.items():
        cmd += ['--set', f'{key}={json.dumps(value)}']
    subprocess.run(cmd, check=True)
    if not (Path(local_output) / 'final/adapter_config.json').is_file():
        raise FileNotFoundError('Training finished without a final adapter; nothing uploaded')
    upload_run(local_output, cloud_subdir)


## SFT only · ms-swift + PEFT
Default PREDICT_WITH_GENERATE=False: monitor teacher-forced loss and token accuracy.
Full training runs 3 epochs and exports the best validation checkpoint as final/.
True: native ms-swift generation evaluation with mmBERT P/R/F1 plugin; select best by F1.
Generation evaluation replaces the teacher-forced validation pass; it does not report val loss simultaneously.
Edit SFT_OVERRIDES for prompt and training parameters. For same-stage resume, set
RESUME_CHECKPOINT to checkpoint-N only if you manually backed it up to Drive; automatic uploads contain final adapters only.


In [ ]:
SFT_OVERRIDES = {}  # e.g. {'training.learning_rate': 0.0001, 'prompt_file': 'prompts/qa.txt'}
RESUME_CHECKPOINT = ''  # manually backed-up checkpoint-N only; final/ is not a resumable checkpoint
sft_args = {**common, 'training.output_dir': SFT}
if not RUN_FULL:
    sft_args.update({'data.limit': 8, 'training.max_steps': 1, 'training.eval_strategy': 'no',
                     'training.load_best_model_at_end': False, 'training.save_steps': 1})
if RESUME_CHECKPOINT:
    copy(cloud(f'task6-runs/{RUN_NAME}/{SFT_STAGE}'), SFT)
    sft_args['resume_from_checkpoint'] = str(Path(SFT) / RESUME_CHECKPOINT)
sft_args.update(SFT_OVERRIDES)
run('mmcultureqa.training.sft', 'configs/sft.yaml', sft_args, SFT, f'task6-runs/{RUN_NAME}/{SFT_STAGE}')


In [ ]:
if not (Path(SFT) / 'final/adapter_config.json').is_file():
    raise FileNotFoundError('No final SFT adapter was saved')
print('SFT adapter on Drive:', cloud(f'task6-runs/{RUN_NAME}/{SFT_STAGE}/final'))
print('Notebook 03: select the same ROOT_FOLDER_ID, SFT_RUN_NAME and INPUT_SFT_STAGE.')


## Learning curves before final evaluation
Plot native training/validation metrics against optimizer step, then upload the PNG to Drive.
Validation points appear only at completed evaluation intervals; no interpolation or smoothing.


In [ ]:
from mmcultureqa.training.curves import plot_learning_curves
plot_learning_curves(SFT, 'sft')
upload_run(SFT, f'task6-runs/{RUN_NAME}/{SFT_STAGE}')


## Final held-out validation score
Generate answers with the saved final adapter and score with the unchanged official scorer.
Default: ALL 1,000 validation examples, even after a one-step training smoke run.
Set VAL_LIMIT=4 only to smoke-test evaluation; subset results are explicitly marked.
BERTScore-F1 is the ranking metric, not exact-match accuracy. No weights are updated.


In [ ]:
VAL_LIMIT = None  # full validation; optionally 4 for an evaluation smoke test
validation_run = Path(SFT)
cmd = [sys.executable, '-m', 'mmcultureqa.evaluate', '--run-dir', str(validation_run)]
cmd += ['--backend', 'transformers' if USE_4BIT else 'vllm']
if VAL_LIMIT is not None:
    cmd += ['--limit', str(VAL_LIMIT)]
result = subprocess.run(cmd)
upload_run(validation_run, f'task6-runs/{RUN_NAME}/{SFT_STAGE}')
result.check_returncode()
